# 1D CNN

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

# Đọc dữ liệu từ file Excel
df = pd.read_excel("data_nckh.xlsx", parse_dates=["Time"])
df = df.sort_values("Time")  # Sắp xếp theo thời gian

# Chuẩn hóa dữ liệu
scaler = MinMaxScaler()
feature_cols = ["CPI_VN", "Gold_Price_World", "Oil_Price", "Tỷ giá USD/VND", "VN-Index",
                "CPI_USA", "SM_M2", "SX_CN", "INTEREST_RATE", "GSI", "GSI_VN"]

df[feature_cols] = scaler.fit_transform(df[feature_cols])
df["Gold_Price_VN"] = scaler.fit_transform(df[["Gold_Price_VN"]])

# Xây dựng chuỗi thời gian với bước trượt (window_size = 30 ngày)
window_size = 30

def create_sequences(data, target_col, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data.iloc[i:i+window_size][feature_cols].values)
        y.append(data.iloc[i+window_size][target_col])
    return np.array(X), np.array(y)

# Chuẩn bị dữ liệu
X, y = create_sequences(df, "Gold_Price_VN", window_size)

# Chia tập train/test
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Best Parameters từ Bayesian Optimization (ĐÃ CẢI TIẾN)
best_params = {
    "dense_units": 64,
    "dropout_rate": 0.3,
    "filters": 16,
    "kernel_size": 2,
    "learning_rate": 0.0011485251581862216
}

# Hàm xây dựng mô hình CNN với best parameters
def build_cnn():
    model = Sequential([
        Input(shape=(30, 11)),  # Định nghĩa đầu vào
        Conv1D(filters=best_params["filters"], kernel_size=best_params["kernel_size"], activation='relu'),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(best_params["dense_units"], activation='relu', kernel_regularizer=l2(0.001)),  # Thêm L2 Regularization
        Dropout(best_params["dropout_rate"]),
        Dense(1, activation='linear')  # Dự đoán giá vàng (hồi quy)
    ])

    optimizer = keras.optimizers.Adam(learning_rate=best_params["learning_rate"])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

# Xây dựng mô hình với best parameters
model = build_cnn()

# Thiết lập Early Stopping để tránh overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)  # Tăng patience lên 15

# Huấn luyện mô hình
history = model.fit(
    X_train, y_train,
    epochs=100,  # Số epochs tối đa
    batch_size=32,
    validation_split=0.2,  # 20% dữ liệu train dùng để validation
    callbacks=[early_stopping],
    verbose=1  # Hiển thị quá trình huấn luyện
)

# Dự đoán trên tập kiểm tra
y_pred = model.predict(X_test)

# Đánh giá mô hình
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

# In kết quả đánh giá
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")


Epoch 1/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.1214 - mae: 0.1454 - val_loss: 0.0467 - val_mae: 0.0471
Epoch 2/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0446 - mae: 0.0636 - val_loss: 0.0273 - val_mae: 0.0347
Epoch 3/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0274 - mae: 0.0513 - val_loss: 0.0173 - val_mae: 0.0274
Epoch 4/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0180 - mae: 0.0422 - val_loss: 0.0121 - val_mae: 0.0248
Epoch 5/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0135 - mae: 0.0426 - val_loss: 0.0090 - val_mae: 0.0280
Epoch 6/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0100 - mae: 0.0378 - val_loss: 0.0063 - val_mae: 0.0223
Epoch 7/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0080 - mae: 0.0378 - val_loss: 0.0051 - val_mae: 0.0228
Epoch 8/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0066 - mae: 0.0351 - val_loss: 0.0039 - val_mae: 0.0201
Epoch 9/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms

# ARIMA

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Đọc dữ liệu
df = pd.read_excel('data_nckh.xlsx')

# Chuyển cột "Time" thành datetime
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y')

df = df.sort_values('Time').reset_index(drop=True)
df.set_index('Time', inplace=True)
# Lấy các biến độc lập (features) và biến mục tiêu (Gold_Price_VN)
target_col = "Gold_Price_VN"
feature_cols = ['CPI_VN', 'Gold_Price_World', 'Oil_Price', 'Tỷ giá USD/VND', 'SX_CN', 
            'VN-Index', 'CPI_USA', 'SM_M2', 'INTEREST_RATE', 'GSI', 'GSI_VN']

# Chuẩn hóa dữ liệu bằng StandardScaler
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
df[target_col] = scaler.fit_transform(df[[target_col]])

# Chuẩn bị dữ liệu cho mô hình
features = df[feature_cols].values
target = df[target_col].values

# Phân chia dữ liệu thành training và test set
X_train, X_test, y_train, y_test = train_test_split(features, target, shuffle=False, test_size=0.2)

# Chia tập train-test (80% train, 20% test) không xáo trộn
train_size = int(0.8 * len(features))
X_train, X_test = features[:train_size], features[train_size:]
y_train, y_test = target[:train_size], target[train_size:]

# Xây dựng mô hình SARIMAX với các biến độc lập (exog)
model = SARIMAX(y_train, exog=X_train, order=(3,1,3),seasonal_order=(3,1,3,12))
model_fit = model.fit()

# Dự báo với mô hình SARIMAX
y_pred_sarimax = model_fit.predict(start=len(y_train), end=len(y_train)+len(y_test)-1, exog=X_test)

# Đánh giá mô hình với các chỉ số như RMSE, MAE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_sarimax))
mae = mean_absolute_error(y_test, y_pred_sarimax)
r2 = r2_score(y_test, y_pred_sarimax)

print("R2_score:", r2)
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")

c:\Users\Admin\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


R2_score: 0.7676927967690655
RMSE: 0.23415812141860454
MAE: 0.1933316501032859


# Stacking-LR (ARIMA & 1D CNN)

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression

# Ensure both predictions have the same number of rows
min_length = min(len(y_pred_sarimax), len(y_pred))
y_pred_sarimax = y_pred_sarimax[:min_length]
y_pred = y_pred[:min_length]
y_test = y_test[:min_length]

# Kết hợp kết quả từ mô hình CNN và ARIMA
stacked_predictions = np.column_stack((y_pred_sarimax, y_pred))

# Huấn luyện mô hình Linear Regression trên kết quả đầu ra của CNN và ARIMA
lr_meta_model = LinearRegression()
lr_meta_model.fit(stacked_predictions, y_test)

# Dự đoán cuối cùng từ mô hình meta (Linear Regression)
y_pred_stacking = lr_meta_model.predict(stacked_predictions)

# Đánh giá các mô hình
r2_stacking = r2_score(y_test, y_pred_stacking)
rmse_stacking = np.sqrt(mean_squared_error(y_test, y_pred_stacking))
mae_stacking = mean_absolute_error(y_test, y_pred_stacking)

# In kết quả đánh giá cho mô hình Stacking
print("Stacking Model – R^2: %.4f, RMSE: %.4f, MAE: %.4f" % (
    r2_stacking, rmse_stacking, mae_stacking
))


Stacking Model – R^2: 0.9608, RMSE: 0.0459, MAE: 0.0197


# Stacking-XGBoost (ARIMA & 1D CNN)

In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

# Ensure both predictions have the same number of rows
min_length = min(len(y_pred_sarimax), len(y_pred))
y_pred_sarimax = y_pred_sarimax[:min_length]
y_pred = y_pred[:min_length]
y_test = y_test[:min_length]

# Kết hợp kết quả từ ARIMA và LSTM
stacked_predictions = np.column_stack((y_pred_sarimax, y_pred))

# Tạo mô hình XGBoost (Boosting)
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', 
                             colsample_bytree=0.89109029727093, 
                             learning_rate=0.07000192276253926, 
                             max_depth=8, 
                             alpha=0, 
                             n_estimators=112,
                             reg_lambda=1,
                             subsample=0.7403725339596576)

# Huấn luyện mô hình XGBoost
xgb_model.fit(stacked_predictions, y_test)

# Dự đoán từ mô hình XGBoost
y_pred_xgb = xgb_model.predict(stacked_predictions)

# Đánh giá mô hình Boosting (XGBoost)
r2_xgb = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

# In kết quả đánh giá cho mô hình XGBoost
print("XGBoost Model – R^2: %.4f, RMSE: %.4f, MAE: %.4f" % (
    r2_xgb, rmse_xgb, mae_xgb
))


XGBoost Model – R^2: 0.9772, RMSE: 0.0350, MAE: 0.0185


# CRNNs

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# Đọc dữ liệu và sắp xếp theo thời gian
df = pd.read_excel("data_nckh.xlsx", parse_dates=["Time"])
df = df.sort_values("Time")

target_col = "Gold_Price_VN"
feature_cols = ["CPI_VN", "Gold_Price_World", "Oil_Price", "Tỷ giá USD/VND", "VN-Index", 
                "CPI_USA", "SM_M2", "SX_CN", "INTEREST_RATE", "GSI", "GSI_VN"]

# Chuẩn hóa dữ liệu
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
df[feature_cols] = feature_scaler.fit_transform(df[feature_cols])
df[target_col] = target_scaler.fit_transform(df[target_col].values.reshape(-1, 1))

# Chuẩn bị dữ liệu cho mô hình
features = df[feature_cols].values
target = df[target_col].values

# Tạo dữ liệu train và test
X_train, X_test, y_train, y_test = train_test_split(features, target, shuffle=False, test_size=0.2)

# Hàm tạo chuỗi thời gian
def create_sequences(X, y, time_step=30):
    X_seq, y_seq = [], []
    for i in range(time_step, len(X)):
        X_seq.append(X[i-time_step:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train, y_train, 30)
X_test_seq, y_test_seq = create_sequences(np.vstack([X_train[-30:], X_test]), 
                                          np.concatenate([y_train[-30:], y_test]), 30)

# CRNN model
crnn_model = models.Sequential([
    layers.Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(30, X_train.shape[1])),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(filters=128, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.LSTM(100, activation='tanh', return_sequences=True),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.LSTM(100, activation='tanh', return_sequences=False),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(50, activation='relu', kernel_regularizer=l2(0.001)),
    layers.Dense(1)
])
# Thiết lập Early Stopping để tránh overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
crnn_model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])

# Huấn luyện mô hình
history = crnn_model.fit(X_train_seq, y_train_seq, epochs=90, batch_size=16, validation_split=0.2, verbose=1)

# Dự báo bằng CRNN trên tập test
y_pred_crnn = crnn_model.predict(X_test_seq).flatten()


# Đánh giá mô hình
print("CRNN – R^2: %.9f, RMSE: %.5f, MAE: %.5f" % (
    r2_score(y_test_seq, y_pred_crnn), 
    np.sqrt(mean_squared_error(y_test_seq, y_pred_crnn)), 
    mean_absolute_error(y_test_seq, y_pred_crnn)
))

c:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 8.5504e-04 - loss: 0.8848 - val_accuracy: 0.0000e+00 - val_loss: 0.1417
Epoch 2/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.0014 - loss: 0.1751 - val_accuracy: 0.0000e+00 - val_loss: 0.1102
Epoch 3/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 5.7964e-04 - loss: 0.1184 - val_accuracy: 0.0000e+00 - val_loss: 0.0687
Epoch 4/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 5.0157e-04 - loss: 0.0908 - val_accuracy: 0.0000e+00 - val_loss: 0.0781
Epoch 5/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 7.5241e-04 - loss: 0.0736 - val_accuracy: 0.0000e+00 - val_loss: 0.0610
Epoch 6/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.0010 - loss: 0.0650 - val_accuracy: 0.0000e+00 - val_loss: 0.0546
Epoch 7/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.0014 - loss: 0.0591 - val_accuracy: 0.0000e+00 - val_loss: 0.0610
Epoch 8/90
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accu

# Stacking-LR (ARIMA & CRNNs)

In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression

# Ensure both predictions have the same number of rows
min_length = min(len(y_pred_sarimax), len(y_pred_crnn))
y_pred_sarimax = y_pred_sarimax[:min_length]
y_pred_crnn = y_pred_crnn[:min_length]
y_test = y_test[:min_length]

# Kết hợp kết quả từ mô hình CNN và ARIMA
stacked_predictions = np.column_stack((y_pred_sarimax, y_pred_crnn))

# Huấn luyện mô hình Linear Regression trên kết quả đầu ra của CNN và ARIMA
lr_meta_model = LinearRegression()
lr_meta_model.fit(stacked_predictions, y_test)

# Dự đoán cuối cùng từ mô hình meta (Linear Regression)
y_pred_stacking = lr_meta_model.predict(stacked_predictions)

# Đánh giá các mô hình
r2_stacking = r2_score(y_test, y_pred_stacking)
rmse_stacking = np.sqrt(mean_squared_error(y_test, y_pred_stacking))
mae_stacking = mean_absolute_error(y_test, y_pred_stacking)

# In kết quả đánh giá cho mô hình Stacking
print("Stacking Model – R^2: %.4f, RMSE: %.4f, MAE: %.4f" % (
    r2_stacking, rmse_stacking, mae_stacking
))


Stacking Model – R^2: 0.9554, RMSE: 0.0488, MAE: 0.0182


# Stacking-XGBoost (ARIMA & CRNNs)

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb


# Ensure both predictions have the same number of rows
min_length = min(len(y_pred_sarimax), len(y_pred_crnn))
y_pred_sarimax = y_pred_sarimax[:min_length]
y_pred_crnn = y_pred_crnn[:min_length]
y_test = y_test[:min_length]

# Kết hợp kết quả từ mô hình CNN và ARIMA
stacked_predictions = np.column_stack((y_pred_sarimax, y_pred_crnn))

# Tạo mô hình XGBoost (Boosting)
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', 
                             colsample_bytree=0.89109029727093, 
                             learning_rate=0.07000192276253926, 
                             max_depth=8, 
                             alpha=0, 
                             n_estimators=112,
                             reg_lambda=1,
                             subsample=0.7403725339596576)

# Huấn luyện mô hình XGBoost
xgb_model.fit(stacked_predictions, y_test)

# Dự đoán từ mô hình XGBoost
y_pred_xgb = xgb_model.predict(stacked_predictions)

# Đánh giá mô hình Boosting (XGBoost)
r2_xgb = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

# In kết quả đánh giá cho mô hình XGBoost
print("XGBoost Model – R^2: %.4f, RMSE: %.4f, MAE: %.4f" % (
    r2_xgb, rmse_xgb, mae_xgb
))

XGBoost Model – R^2: 0.9703, RMSE: 0.0398, MAE: 0.0189


# LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models

# Đọc dữ liệu 14 năm từ file Excel
df = pd.read_excel("data_nckh.xlsx", parse_dates=["Time"])
df = df.sort_values("Time")  # sắp xếp theo thời gian

# Kiểm tra và xử lý dữ liệu missing/duplicate
assert df["Time"].is_unique, "Thời gian bị trùng lặp!"  # đảm bảo không trùng
assert df.isnull().sum().sum() == 0, "Có giá trị null trong dữ liệu!"  # đảm bảo không có null

# Lấy cột giá vàng Việt Nam làm biến mục tiêu, các cột khác làm đặc trưng
target_col = "Gold_Price_VN"
feature_cols = ["CPI_VN", "Gold_Price_World", "Oil_Price", "Tỷ giá USD/VND", "VN-Index", 
                "CPI_USA", "SM_M2", "SX_CN", "INTEREST_RATE", "GSI", "GSI_VN"]

# Chuẩn hóa dữ liệu bằng StandardScaler
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
df[target_col] = scaler.fit_transform(df[[target_col]])

# Chuẩn bị dữ liệu cho mô hình
features = df[feature_cols].values
target = df[target_col].values

# Tính toán chỉ số phân chia
train_size = int(0.8 * len(features))  # 80% dữ liệu đầu tiên

# Chia dữ liệu mà không xáo trộn
X_train, X_test = features[:train_size], features[train_size:]
y_train, y_test = target[:train_size], target[train_size:]


# --------------------------------------
# Tạo dữ liệu chuỗi cho LSTM
time_step = 30  # độ dài chuỗi đầu vào cho LSTM

def create_sequences(X, y, time_step=30):
    X_seq, y_seq = [], []
    for i in range(time_step, len(X)):
        X_seq.append(X[i-time_step:i])    # đoạn [i-time_step ... i-1]
        y_seq.append(y[i])                # giá trị tại thời điểm i (dự báo ở bước kế)
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train, y_train, time_step)
X_test_seq, y_test_seq = create_sequences(np.vstack([X_train[-time_step:], X_test]), 
                                         np.concatenate([y_train[-time_step:], y_test]), 
                                         time_step)

# Xây dựng mô hình LSTM với nhiều layer hơn và BatchNormalization
lstm_model = models.Sequential([
    layers.LSTM(100, activation='tanh', return_sequences=True, input_shape=(30, X_train.shape[1])),
    layers.BatchNormalization(),
    layers.Dropout(0.29641518435845243),

    layers.LSTM(200, activation='tanh', return_sequences=True),
    layers.BatchNormalization(),
    layers.Dropout(0.4563329965439854),

    layers.LSTM(200, activation='tanh', return_sequences=False),
    layers.BatchNormalization(),
    layers.Dropout(0.36815419512523484),

    layers.Dense(50, activation='relu'),
    layers.Dense(1)
]) 

# Biên dịch mô hình với việc theo dõi accuracy
lstm_model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])

# Huấn luyện mô hình và lưu lịch sử
history = lstm_model.fit(X_train_seq, y_train_seq, epochs=90, batch_size=16, validation_split=0.1, verbose=1)

# Dự đoán và tính các chỉ số
y_pred_lstm = lstm_model.predict(X_test_seq).flatten()
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))

# Đánh giá mô hình LSTM
print("LSTM – R^2: %.9f, RMSE: %.5f, MAE: %.5f" % (
    r2_score(y_test, y_pred_lstm), 
    rmse_lstm, 
    mean_absolute_error(y_test, y_pred_lstm)
))

c:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 17s 40ms/step - accuracy: 2.9704e-04 - loss: 0.7707 - val_accuracy: 0.0000e+00 - val_loss: 0.0483
Epoch 2/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 3.8912e-05 - loss: 0.1616 - val_accuracy: 0.0000e+00 - val_loss: 0.0532
Epoch 3/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - accuracy: 0.0021 - loss: 0.0760 - val_accuracy: 0.0000e+00 - val_loss: 0.0189
Epoch 4/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 6.2633e-04 - loss: 0.0387 - val_accuracy: 0.0000e+00 - val_loss: 0.0071
Epoch 5/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 9.1679e-04 - loss: 0.0231 - val_accuracy: 0.0000e+00 - val_loss: 0.0046
Epoch 6/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 4.1877e-04 - loss: 0.0153 - val_accuracy: 0.0000e+00 - val_loss: 0.0034
Epoch 7/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 1.6578e-04 - loss: 0.0135 - val_accuracy: 0.0000e+00 - val_loss: 0.0039
Epoch 8/90
245/245 ━━━━━━━━━━━━━━━━━━━━ 9

# Stacking-LR (ARIMA & LSTM)

In [18]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression

# Ensure both predictions have the same number of rows
min_length = min(len(y_pred_sarimax), len(y_pred_lstm))
y_pred_sarimax = y_pred_sarimax[:min_length]
y_pred_lstm = y_pred_lstm[:min_length]
y_test = y_test[:min_length]

# Kết hợp kết quả từ mô hình SARIMAX và LSTM
stacked_predictions = np.column_stack((y_pred_sarimax, y_pred_lstm))

# Huấn luyện mô hình Linear Regression trên kết quả đầu ra của CNN và ARIMA
lr_meta_model = LinearRegression()
lr_meta_model.fit(stacked_predictions, y_test)

# Dự đoán cuối cùng từ mô hình meta (Linear Regression)
y_pred_stacking = lr_meta_model.predict(stacked_predictions)

# Đánh giá các mô hình
r2_stacking = r2_score(y_test, y_pred_stacking)
rmse_stacking = np.sqrt(mean_squared_error(y_test, y_pred_stacking))
mae_stacking = mean_absolute_error(y_test, y_pred_stacking)

# In kết quả đánh giá cho mô hình Stacking
print("Stacking Model – R^2: %.4f, RMSE: %.4f, MAE: %.4f" % (
    r2_stacking, rmse_stacking, mae_stacking
))


Stacking Model – R^2: 0.9851, RMSE: 0.0282, MAE: 0.0138


# Stacking-XGBoost (ARIMA & LSTM)

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb


# Ensure both predictions have the same number of rows
min_length = min(len(y_pred_sarimax), len(y_pred_lstm))
y_pred_sarimax = y_pred_sarimax[:min_length]
y_pred_lstm = y_pred_lstm[:min_length]
y_test = y_test[:min_length]

# Kết hợp kết quả từ mô hình CNN và ARIMA
stacked_predictions = np.column_stack((y_pred_sarimax, y_pred_lstm))

# Tạo mô hình XGBoost (Boosting)
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', 
                             colsample_bytree=0.89109029727093, 
                             learning_rate=0.07000192276253926, 
                             max_depth=8, 
                             alpha=0, 
                             n_estimators=112,
                             reg_lambda=1,
                             subsample=0.7403725339596576)

# Huấn luyện mô hình XGBoost
xgb_model.fit(stacked_predictions, y_test)

# Dự đoán từ mô hình XGBoost
y_pred_xgb = xgb_model.predict(stacked_predictions)

# Đánh giá mô hình Boosting (XGBoost)
r2_xgb = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

# In kết quả đánh giá cho mô hình XGBoost
print("XGBoost Model – R^2: %.4f, RMSE: %.4f, MAE: %.4f" % (
    r2_xgb, rmse_xgb, mae_xgb
))

XGBoost Model – R^2: 0.9908, RMSE: 0.0222, MAE: 0.0132
